
# FairWarn-SHS — Notebook 10B
## OULAD Graph External Validation

This notebook performs the **graph-specific public validation** using the
Open University Learning Analytics Dataset (OULAD).

### Important design choices

- OULAD is kept completely separate from the Ghana field dataset.
- No Ghana students or Ghana edges are used here.
- The target is derived from OULAD's `final_result`.
- Only **early VLE activity from days 0–28** is used for behavioural features.
- No final grades or assessment scores are used.
- A student graph is constructed from the early interaction logs using a
  transparent k-nearest-neighbour similarity rule.
- The same node features are supplied to a feature-only MLP and GraphSAGE.
- Therefore, the comparison asks whether the interaction-derived graph adds
  predictive information beyond the node features.

### OULAD source

Kuzilek, Hlosta, and Zdrahal (2017), Open University Learning Analytics Dataset.
Scientific Data, 4, 170171.
DOI: 10.1038/sdata.2017.171


In [ ]:
!pip -q install torch-geometric pandas numpy scikit-learn matplotlib scipy


## Step 1 — Download the official OULAD ZIP

The download is from the Open University's official OULAD hosting.

The ZIP is kept inside the Colab session and is **not** mixed with the Ghana data.


In [ ]:

from pathlib import Path
import urllib.request

OULAD_URL = "https://schools.stem.open.ac.uk/cdn/files/anonymisedData.zip"
ZIP_PATH = Path("anonymisedData.zip")

if not ZIP_PATH.exists():
    print("Downloading OULAD. This can take several minutes...")
    urllib.request.urlretrieve(OULAD_URL, ZIP_PATH)
    print("Download complete.")
else:
    print("OULAD ZIP already exists in this Colab session.")

print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / (1024**2), 1))


In [ ]:

import json
import random
import zipfile
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, normalize
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    brier_score_loss,
)
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

SEEDS = [42, 123, 456, 789, 1010]
EARLY_WINDOW_START = 0
EARLY_WINDOW_END = 28
K_NEIGHBORS = 5

OUTPUT_DIR = Path("outputs/public_validation/oulad")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



## Step 2 — Inspect the official OULAD tables

OULAD is not one flat CSV. It contains linked tables.

We use:

- `studentInfo.csv` — student characteristics and final result;
- `vle.csv` — information about VLE resources;
- `studentVle.csv` — dated student interactions/clicks with VLE resources.

The notebook chooses **one module-presentation deterministically**: the
module-presentation with the largest number of students in `studentInfo`.
This keeps the graph manageable and makes the selection rule reproducible.


In [ ]:

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    names = archive.namelist()
    print("Files in OULAD ZIP:")
    for name in names:
        print(" -", name)

    with archive.open("studentInfo.csv") as file:
        student_info = pd.read_csv(file)

    with archive.open("vle.csv") as file:
        vle = pd.read_csv(file)

print()
print("studentInfo shape:", student_info.shape)
print("vle shape:", vle.shape)
print("studentInfo columns:", student_info.columns.tolist())


In [ ]:

presentation_counts = (
    student_info
    .groupby(["code_module", "code_presentation"])
    .size()
    .reset_index(name="Students")
    .sort_values(
        ["Students", "code_module", "code_presentation"],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)

display(presentation_counts)

SELECTED_MODULE = presentation_counts.loc[0, "code_module"]
SELECTED_PRESENTATION = presentation_counts.loc[0, "code_presentation"]

print()
print("Selected module:", SELECTED_MODULE)
print("Selected presentation:", SELECTED_PRESENTATION)
print("Students:", int(presentation_counts.loc[0, "Students"]))



## Step 3 — Define the public at-risk label

OULAD's `final_result` contains outcomes such as:

- Pass
- Distinction
- Fail
- Withdrawn

For this external early-warning experiment:

```text
At risk (1)     = Fail or Withdrawn
Not at risk (0) = Pass or Distinction
```

This is an OULAD-specific target. It is not claimed to be identical to the
Ghana SHS target.


In [ ]:

cohort = student_info[
    student_info["code_module"].eq(SELECTED_MODULE)
    & student_info["code_presentation"].eq(SELECTED_PRESENTATION)
].copy()

cohort["TARGET_AtRisk"] = cohort["final_result"].isin(
    ["Fail", "Withdrawn"]
).astype(int)

print("Cohort size:", len(cohort))
print()
print("Final-result counts:")
print(cohort["final_result"].value_counts(dropna=False))
print()
print("At-risk:", int(cohort["TARGET_AtRisk"].sum()))
print("Not-at-risk:", int((cohort["TARGET_AtRisk"] == 0).sum()))
print("At-risk rate:", round(cohort["TARGET_AtRisk"].mean(), 4))



## Step 4 — Read only the first 28 days of VLE activity

`studentVle.csv` is large, so it is read in chunks.

We retain only:

- the selected module;
- the selected presentation;
- dates from day 0 through day 28.

This keeps the experiment genuinely early.


In [ ]:

early_chunks = []

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    with archive.open("studentVle.csv") as file:
        for chunk_number, chunk in enumerate(
            pd.read_csv(file, chunksize=500_000),
            start=1
        ):
            keep = chunk[
                chunk["code_module"].eq(SELECTED_MODULE)
                & chunk["code_presentation"].eq(SELECTED_PRESENTATION)
                & chunk["date"].between(
                    EARLY_WINDOW_START,
                    EARLY_WINDOW_END
                )
            ].copy()

            if not keep.empty:
                early_chunks.append(keep)

            if chunk_number % 5 == 0:
                print("Processed chunks:", chunk_number)

early_vle = pd.concat(
    early_chunks,
    ignore_index=True
)

print()
print("Early interaction rows:", len(early_vle))
print("Students with early interactions:", early_vle["id_student"].nunique())
print("Date range:", early_vle["date"].min(), "to", early_vle["date"].max())



## Step 5 — Build early behavioural features

For every student we calculate:

- total clicks;
- number of active days;
- number of unique VLE resources;
- mean clicks per active day;
- clicks by VLE activity type.

Students with no activity in days 0–28 receive zero activity values.


In [ ]:

selected_vle = vle[
    vle["code_module"].eq(SELECTED_MODULE)
    & vle["code_presentation"].eq(SELECTED_PRESENTATION)
][
    [
        "id_site",
        "activity_type",
    ]
].copy()

early_vle = early_vle.merge(
    selected_vle,
    on="id_site",
    how="left"
)

base_activity = (
    early_vle
    .groupby("id_student")
    .agg(
        total_clicks=("sum_click", "sum"),
        active_days=("date", "nunique"),
        unique_sites=("id_site", "nunique"),
    )
    .reset_index()
)

base_activity["mean_clicks_per_active_day"] = (
    base_activity["total_clicks"]
    / base_activity["active_days"].replace(0, np.nan)
).fillna(0.0)

activity_pivot = (
    early_vle
    .pivot_table(
        index="id_student",
        columns="activity_type",
        values="sum_click",
        aggfunc="sum",
        fill_value=0,
    )
    .add_prefix("clicks_")
    .reset_index()
)

activity_features = base_activity.merge(
    activity_pivot,
    on="id_student",
    how="outer"
)

cohort_features = cohort.merge(
    activity_features,
    on="id_student",
    how="left"
)

activity_columns = [
    c for c in cohort_features.columns
    if c.startswith("clicks_")
    or c in {
        "total_clicks",
        "active_days",
        "unique_sites",
        "mean_clicks_per_active_day",
    }
]

cohort_features[activity_columns] = (
    cohort_features[activity_columns]
    .fillna(0)
)

print("Activity feature count:", len(activity_columns))
print("Students with zero early clicks:", int(
    cohort_features["total_clicks"].eq(0).sum()
))
display(cohort_features[[
    "id_student",
    "TARGET_AtRisk",
    "total_clicks",
    "active_days",
    "unique_sites",
    "mean_clicks_per_active_day",
]].head())



## Step 6 — Construct the interaction-derived student graph

OULAD does not provide friendship or peer-study edges.

We therefore **do not claim these are social relationships**.

Instead, each student is connected to the five students with the most similar
first-28-day VLE activity profiles, measured using cosine distance.

This is a reproducible transformation of genuine interaction logs:

```text
early VLE interactions
        ↓
activity profile per student
        ↓
cosine similarity
        ↓
5-nearest-neighbour student graph
```

No outcome labels are used to create the edges.


In [ ]:

graph_profile_columns = activity_columns.copy()

graph_profiles = cohort_features[
    graph_profile_columns
].astype(float).to_numpy()

# Normalise profiles for cosine similarity.
graph_profiles_normalised = normalize(
    graph_profiles,
    norm="l2"
)

n_neighbors = min(
    K_NEIGHBORS + 1,
    len(cohort_features)
)

knn = NearestNeighbors(
    n_neighbors=n_neighbors,
    metric="cosine",
    algorithm="brute",
)

knn.fit(graph_profiles_normalised)

distances, neighbours = knn.kneighbors(
    graph_profiles_normalised
)

undirected_edges = set()

for source_index in range(len(cohort_features)):
    for target_index in neighbours[source_index]:
        if source_index == target_index:
            continue

        edge = tuple(sorted(
            (int(source_index), int(target_index))
        ))
        undirected_edges.add(edge)

edge_pairs = []

for source_index, target_index in sorted(undirected_edges):
    edge_pairs.extend([
        (source_index, target_index),
        (target_index, source_index),
    ])

edge_index = torch.tensor(
    edge_pairs,
    dtype=torch.long
).t().contiguous()

degrees = torch.bincount(
    edge_index[0],
    minlength=len(cohort_features)
)

print("Student nodes:", len(cohort_features))
print("Unique undirected edges:", len(undirected_edges))
print("Mean degree:", round(float(degrees.float().mean()), 2))
print("Minimum degree:", int(degrees.min()))
print("Maximum degree:", int(degrees.max()))



## Step 7 — Prepare node features

The predictive node features use:

- OULAD demographic/study-background variables available in `studentInfo`;
- first-28-day VLE activity features.

The target and final result are excluded.

The same node feature set is used for both the MLP and GraphSAGE.


In [ ]:

excluded_predictors = {
    "code_module",
    "code_presentation",
    "id_student",
    "final_result",
    "TARGET_AtRisk",
}

candidate_columns = [
    c for c in cohort_features.columns
    if c not in excluded_predictors
]

# registration/unregistration dates can encode withdrawal timing, so exclude them.
candidate_columns = [
    c for c in candidate_columns
    if c not in {
        "date_registration",
        "date_unregistration",
    }
]

node_feature_columns = candidate_columns

print("Node feature count:", len(node_feature_columns))
print("Node features:")
for column in node_feature_columns:
    print(" -", column)


In [ ]:

def build_preprocessor(frame):
    numeric_columns = frame.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    categorical_columns = [
        c for c in frame.columns
        if c not in numeric_columns
    ]

    return ColumnTransformer([
        (
            "numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    ),
                ),
                (
                    "scaler",
                    StandardScaler()
                ),
            ]),
            numeric_columns,
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    ),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical_columns,
        ),
    ])


def calculate_metrics(
    y_true,
    probability,
    prediction,
):
    return {
        "AUC_ROC": roc_auc_score(
            y_true,
            probability,
        ),
        "AUC_PR": average_precision_score(
            y_true,
            probability,
        ),
        "Precision_AtRisk": precision_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "Recall_AtRisk": recall_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "F1_AtRisk": f1_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "Weighted_F1": f1_score(
            y_true,
            prediction,
            average="weighted",
            zero_division=0,
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true,
            prediction,
        ),
        "Accuracy": accuracy_score(
            y_true,
            prediction,
        ),
        "Brier_Score": brier_score_loss(
            y_true,
            probability,
        ),
    }



## Step 8 — Run the feature-only MLP and GraphSAGE

For every seed:

- 65% training;
- 15% validation;
- 20% held-out test;
- stratified by at-risk status.

The preprocessing transformer is fitted on the training nodes only and then
applied to validation/test nodes.


In [ ]:

class OULADGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = SAGEConv(
            in_channels,
            64,
            aggr="mean",
        )
        self.conv2 = SAGEConv(
            64,
            32,
            aggr="mean",
        )
        self.classifier = torch.nn.Linear(
            32,
            2,
        )
        self.dropout = 0.35

    def forward(self, x, edge_index):
        x = F.relu(
            self.conv1(
                x,
                edge_index,
            )
        )
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training,
        )

        x = F.relu(
            self.conv2(
                x,
                edge_index,
            )
        )
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training,
        )

        return self.classifier(x)


In [ ]:

X_raw = cohort_features[
    node_feature_columns
].copy()

y_public = cohort_features[
    "TARGET_AtRisk"
].astype(int).to_numpy()

seed_rows = []
prediction_rows = []

for seed in SEEDS:
    set_seed(seed)

    all_indices = np.arange(
        len(cohort_features)
    )

    train_val_indices, test_indices = train_test_split(
        all_indices,
        test_size=0.20,
        stratify=y_public,
        random_state=seed,
    )

    train_val_y = y_public[
        train_val_indices
    ]

    train_indices, validation_indices = train_test_split(
        train_val_indices,
        test_size=0.1875,
        stratify=train_val_y,
        random_state=seed,
    )

    preprocessor = build_preprocessor(
        X_raw.iloc[train_indices]
    )

    X_train = preprocessor.fit_transform(
        X_raw.iloc[train_indices]
    ).astype(np.float32)

    X_validation = preprocessor.transform(
        X_raw.iloc[validation_indices]
    ).astype(np.float32)

    X_test = preprocessor.transform(
        X_raw.iloc[test_indices]
    ).astype(np.float32)

    X_all_encoded = preprocessor.transform(
        X_raw
    ).astype(np.float32)

    # ---------- Feature-only MLP ----------
    mlp = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=0.0005,
        learning_rate_init=0.001,
        max_iter=500,
        early_stopping=True,
        random_state=seed,
    )

    mlp.fit(
        X_train,
        y_public[train_indices],
    )

    mlp_probability = mlp.predict_proba(
        X_test
    )[:, 1]

    mlp_prediction = (
        mlp_probability >= 0.5
    ).astype(int)

    mlp_metrics = calculate_metrics(
        y_public[test_indices],
        mlp_probability,
        mlp_prediction,
    )

    seed_rows.append({
        "Dataset": "OULAD",
        "Model": "Feature-only MLP",
        "Seed": seed,
        **mlp_metrics,
    })

    for idx, truth, pred, prob in zip(
        test_indices,
        y_public[test_indices],
        mlp_prediction,
        mlp_probability,
    ):
        prediction_rows.append({
            "Dataset": "OULAD",
            "Model": "Feature-only MLP",
            "Seed": seed,
            "Node_Index": int(idx),
            "id_student": int(
                cohort_features.iloc[idx][
                    "id_student"
                ]
            ),
            "True_Label": int(truth),
            "Predicted_Label": int(pred),
            "AtRisk_Probability": float(prob),
        })

    # ---------- GraphSAGE ----------
    x_tensor = torch.tensor(
        X_all_encoded,
        dtype=torch.float32,
    )

    graph = Data(
        x=x_tensor,
        edge_index=edge_index,
        y=torch.tensor(
            y_public,
            dtype=torch.long,
        ),
    )

    train_mask = torch.zeros(
        len(cohort_features),
        dtype=torch.bool,
    )
    validation_mask = torch.zeros(
        len(cohort_features),
        dtype=torch.bool,
    )
    test_mask = torch.zeros(
        len(cohort_features),
        dtype=torch.bool,
    )

    train_mask[train_indices] = True
    validation_mask[validation_indices] = True
    test_mask[test_indices] = True

    graph.train_mask = train_mask
    graph.validation_mask = validation_mask
    graph.test_mask = test_mask
    graph = graph.to(device)

    model = OULADGraphSAGE(
        graph.num_node_features
    ).to(device)

    train_labels = graph.y[
        graph.train_mask
    ]

    counts = torch.bincount(
        train_labels,
        minlength=2,
    ).float()

    class_weights = (
        counts.sum()
        / (
            2.0
            * counts.clamp_min(1.0)
        )
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.005,
        weight_decay=5e-4,
    )

    best_state = None
    best_validation_ap = -np.inf
    best_epoch = 0
    wait = 0

    for epoch in range(1, 501):
        model.train()
        optimizer.zero_grad()

        logits = model(
            graph.x,
            graph.edge_index,
        )

        loss = F.cross_entropy(
            logits[graph.train_mask],
            graph.y[graph.train_mask],
            weight=class_weights,
        )

        loss.backward()
        optimizer.step()

        model.eval()

        with torch.no_grad():
            logits = model(
                graph.x,
                graph.edge_index,
            )

            probability_all = torch.softmax(
                logits,
                dim=1,
            )[:, 1]

            validation_true = (
                graph.y[
                    graph.validation_mask
                ]
                .cpu()
                .numpy()
            )

            validation_probability = (
                probability_all[
                    graph.validation_mask
                ]
                .cpu()
                .numpy()
            )

            validation_ap = average_precision_score(
                validation_true,
                validation_probability,
            )

        if (
            validation_ap
            > best_validation_ap + 1e-6
        ):
            best_validation_ap = (
                validation_ap
            )
            best_epoch = epoch
            best_state = deepcopy(
                model.state_dict()
            )
            wait = 0
        else:
            wait += 1

        if wait >= 40:
            break

    model.load_state_dict(
        best_state
    )
    model.eval()

    with torch.no_grad():
        logits = model(
            graph.x,
            graph.edge_index,
        )
        probability_all = torch.softmax(
            logits,
            dim=1,
        )[:, 1]
        prediction_all = torch.argmax(
            logits,
            dim=1,
        )

    graph_truth = (
        graph.y[
            graph.test_mask
        ]
        .cpu()
        .numpy()
    )

    graph_probability = (
        probability_all[
            graph.test_mask
        ]
        .cpu()
        .numpy()
    )

    graph_prediction = (
        prediction_all[
            graph.test_mask
        ]
        .cpu()
        .numpy()
    )

    graph_metrics = calculate_metrics(
        graph_truth,
        graph_probability,
        graph_prediction,
    )

    seed_rows.append({
        "Dataset": "OULAD",
        "Model": "GraphSAGE",
        "Seed": seed,
        "Best_Epoch": best_epoch,
        **graph_metrics,
    })

    for idx, truth, pred, prob in zip(
        test_indices,
        graph_truth,
        graph_prediction,
        graph_probability,
    ):
        prediction_rows.append({
            "Dataset": "OULAD",
            "Model": "GraphSAGE",
            "Seed": seed,
            "Node_Index": int(idx),
            "id_student": int(
                cohort_features.iloc[idx][
                    "id_student"
                ]
            ),
            "True_Label": int(truth),
            "Predicted_Label": int(pred),
            "AtRisk_Probability": float(prob),
        })

    print(
        f"Seed {seed} | "
        f"MLP AUC-PR={mlp_metrics['AUC_PR']:.4f}, "
        f"Recall={mlp_metrics['Recall_AtRisk']:.4f} | "
        f"GraphSAGE AUC-PR={graph_metrics['AUC_PR']:.4f}, "
        f"Recall={graph_metrics['Recall_AtRisk']:.4f}"
    )

metrics_by_seed_df = pd.DataFrame(
    seed_rows
)

predictions_df = pd.DataFrame(
    prediction_rows
)



## Step 9 — Summarise the external graph result

The key comparison is:

```text
GraphSAGE
versus
Feature-only MLP
```

because both models receive the same node features. The main difference is that
GraphSAGE can aggregate information from the interaction-derived graph.


In [ ]:

metric_columns = [
    "AUC_ROC",
    "AUC_PR",
    "Precision_AtRisk",
    "Recall_AtRisk",
    "F1_AtRisk",
    "Weighted_F1",
    "Balanced_Accuracy",
    "Accuracy",
    "Brier_Score",
]

summary_rows = []

for model_name, group in metrics_by_seed_df.groupby(
    "Model"
):
    row = {
        "Dataset": "OULAD",
        "Module": SELECTED_MODULE,
        "Presentation": SELECTED_PRESENTATION,
        "Early_Window_Days": f"{EARLY_WINDOW_START}-{EARLY_WINDOW_END}",
        "Graph_K": K_NEIGHBORS,
        "Model": model_name,
        "Seeds": group["Seed"].nunique(),
    }

    for metric in metric_columns:
        row[
            f"{metric}_Mean"
        ] = group[metric].mean()

        row[
            f"{metric}_SD"
        ] = group[metric].std(
            ddof=1
        )

    summary_rows.append(row)

summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    "AUC_PR_Mean",
    ascending=False,
)

summary_df


In [ ]:

mlp_summary = summary_df[
    summary_df["Model"].eq(
        "Feature-only MLP"
    )
].iloc[0]

graph_summary = summary_df[
    summary_df["Model"].eq(
        "GraphSAGE"
    )
].iloc[0]

graph_contribution_df = pd.DataFrame([{
    "Dataset": "OULAD",
    "Module": SELECTED_MODULE,
    "Presentation": SELECTED_PRESENTATION,
    "Delta_AUC_PR_GraphSAGE_Minus_MLP": (
        graph_summary["AUC_PR_Mean"]
        - mlp_summary["AUC_PR_Mean"]
    ),
    "Delta_Recall_GraphSAGE_Minus_MLP": (
        graph_summary["Recall_AtRisk_Mean"]
        - mlp_summary["Recall_AtRisk_Mean"]
    ),
    "Delta_F1_GraphSAGE_Minus_MLP": (
        graph_summary["F1_AtRisk_Mean"]
        - mlp_summary["F1_AtRisk_Mean"]
    ),
    "Delta_Balanced_Accuracy_GraphSAGE_Minus_MLP": (
        graph_summary["Balanced_Accuracy_Mean"]
        - mlp_summary["Balanced_Accuracy_Mean"]
    ),
}])

graph_contribution_df


In [ ]:

plt.figure(figsize=(7, 5))

plot_df = summary_df.set_index(
    "Model"
).reindex([
    "Feature-only MLP",
    "GraphSAGE",
])

plt.bar(
    plot_df.index,
    plot_df["AUC_PR_Mean"],
    yerr=plot_df["AUC_PR_SD"],
    capsize=5,
)

plt.ylabel("OULAD test AUC-PR")
plt.title(
    "External graph validation: MLP vs GraphSAGE"
)
plt.tight_layout()
plt.show()



## How to explain this experiment

A concise explanation is:

> OULAD supplied real dated VLE interaction records rather than social peer links.
> I converted the first 28 days of interaction logs into student activity profiles
> and connected each student to the five students with the most similar early
> activity profile. No outcome label was used when creating the graph. I then
> compared GraphSAGE with a feature-only MLP using the same node features.

This is an **interaction-derived similarity graph**, not a friendship network.
That distinction must be preserved in the thesis.


In [ ]:

source_metadata = {
    "dataset_name": "Open University Learning Analytics Dataset (OULAD)",
    "source": "The Open University",
    "doi": "10.1038/sdata.2017.171",
    "official_download": OULAD_URL,
    "module": str(SELECTED_MODULE),
    "presentation": str(SELECTED_PRESENTATION),
    "cohort_size": int(len(cohort_features)),
    "early_window_start_day": EARLY_WINDOW_START,
    "early_window_end_day": EARLY_WINDOW_END,
    "graph_construction": (
        "Undirected 5-nearest-neighbour student similarity graph "
        "from first-28-day VLE activity profiles using cosine distance."
    ),
    "uses_outcome_labels_for_edges": False,
    "merged_with_ghana_field_data": False,
}

with open(
    OUTPUT_DIR
    / "fairwarn10b_oulad_source_and_graph_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        source_metadata,
        file,
        indent=2,
    )

summary_df.to_csv(
    OUTPUT_DIR
    / "fairwarn10b_oulad_summary_mean_sd.csv",
    index=False,
)

metrics_by_seed_df.to_csv(
    OUTPUT_DIR
    / "fairwarn10b_oulad_metrics_by_seed.csv",
    index=False,
)

graph_contribution_df.to_csv(
    OUTPUT_DIR
    / "fairwarn10b_graph_contribution.csv",
    index=False,
)

predictions_df.to_csv(
    OUTPUT_DIR
    / "fairwarn10b_oulad_predictions.csv",
    index=False,
)

from google.colab import files

files.download(
    str(
        OUTPUT_DIR
        / "fairwarn10b_oulad_summary_mean_sd.csv"
    )
)

files.download(
    str(
        OUTPUT_DIR
        / "fairwarn10b_oulad_metrics_by_seed.csv"
    )
)

files.download(
    str(
        OUTPUT_DIR
        / "fairwarn10b_graph_contribution.csv"
    )
)

files.download(
    str(
        OUTPUT_DIR
        / "fairwarn10b_oulad_source_and_graph_metadata.json"
    )
)
